# 01 Data Preparation and Y-Ranking Split

This notebook prepares the QDB.177 logKoc dataset for QSAR modeling.

Run it in the `mordred` conda environment, where `rdkit` and `pandas` are available. It reads the immutable raw QDB archive under `data/raw/`, validates and canonicalizes SMILES with RDKit, removes disconnected structures and duplicate canonical SMILES, then writes the Y-ranking train/test split to `data/processed/train.csv` and `data/processed/test.csv`.

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd
from rdkit import Chem

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOGKOC_VALUES = RAW_DIR / "properties" / "M2.logKoc" / "values"
COMPOUNDS_DIR = RAW_DIR / "compounds"

assert RAW_DIR.exists(), f"Missing raw data directory: {RAW_DIR}"
assert LOGKOC_VALUES.exists(), f"Missing logKoc values file: {LOGKOC_VALUES}"
assert COMPOUNDS_DIR.exists(), f"Missing compounds directory: {COMPOUNDS_DIR}"


## Raw Archive Orientation

Safe parsing assumption: `data/raw/properties/properties.xml` registers `M2.logKoc` as the soil sorption coefficient endpoint, and `data/raw/properties/M2.logKoc/values` maps `Compound Id` to the logKoc response value. Each compound's SMILES is stored in `data/raw/compounds/<Compound Id>/daylight-smiles`.

The exploratory cell below prints the registered property name and a small sample from both the values file and one SMILES file so the linkage remains auditable.

In [2]:
namespace = {"qdb": "http://www.qsardb.org/QDB"}
properties_xml = RAW_DIR / "properties" / "properties.xml"
properties_tree = ET.parse(properties_xml)
logkoc_property = None
for property_node in properties_tree.findall("qdb:Property", namespace):
    property_id = property_node.findtext("qdb:Id", namespaces=namespace)
    if property_id == "M2.logKoc":
        logkoc_property = {
            "Id": property_id,
            "Name": property_node.findtext("qdb:Name", namespaces=namespace),
            "Description": property_node.findtext("qdb:Description", default="", namespaces=namespace),
        }
        break

assert logkoc_property is not None, "M2.logKoc was not found in properties.xml"
print("Registered target property:")
print(logkoc_property)
print("\nFirst rows of M2.logKoc/values:")
display(pd.read_csv(LOGKOC_VALUES, sep="\t").head())
print("\nExample compound 1 daylight SMILES:")
print((COMPOUNDS_DIR / "1" / "daylight-smiles").read_text().strip())


Registered target property:
{'Id': 'M2.logKoc', 'Name': 'Soil sorption coefficient as log(Koc)', 'Description': 'Soil sorption partition coefficient, expressed as the ratio between chemical concentration in soil and in water, normalized to organic carbon (Koc)'}

First rows of M2.logKoc/values:


,Compound Id,The soil sorption partition coefficient
0,1,1.64
1,4,3.09
2,5,2.61
3,9,3.39
4,51,2.57



Example compound 1 daylight SMILES:
c1(N)nc[nH]n1


## Parse logKoc Values and SMILES

The raw values file is tab-separated. The response file is treated as authoritative for the modeling cohort; compound IDs are then used to locate the matching SMILES files.

In [3]:
def load_logkoc_values(values_path: Path) -> pd.DataFrame:
    df = pd.read_csv(values_path, sep="\t")
    expected_columns = ["Compound Id", "The soil sorption partition coefficient"]
    assert list(df.columns) == expected_columns, f"Unexpected columns: {list(df.columns)}"
    df = df.rename(columns={
        "Compound Id": "compound_id",
        "The soil sorption partition coefficient": "logKoc",
    })
    df["compound_id"] = df["compound_id"].astype(int)
    df["logKoc"] = pd.to_numeric(df["logKoc"], errors="raise")
    return df


def read_daylight_smiles(compound_id: int) -> str:
    smiles_path = COMPOUNDS_DIR / str(compound_id) / "daylight-smiles"
    if not smiles_path.exists():
        raise FileNotFoundError(f"Missing SMILES for compound {compound_id}: {smiles_path}")
    lines = [line.strip() for line in smiles_path.read_text().splitlines() if line.strip()]
    if len(lines) != 1:
        raise ValueError(f"Expected exactly one SMILES line for compound {compound_id}, found {len(lines)}")
    return lines[0]

raw_df = load_logkoc_values(LOGKOC_VALUES)
raw_df["SMILES_raw"] = raw_df["compound_id"].map(read_daylight_smiles)
raw_df = raw_df[["compound_id", "SMILES_raw", "logKoc"]]

print(f"Total raw parsed compounds: {len(raw_df)}")
display(raw_df.head())


Total raw parsed compounds: 643


,compound_id,SMILES_raw,logKoc
0,1,c1(N)nc[nH]n1,1.64
1,4,c1ccc(c2c1n1c(s2)nnc1)C,3.09
2,5,c1c(ccc(c1)O[C@H](C(=O)C(C)(C)C)n1cncn1)Cl,2.61
3,9,n1(ncnc1)C[C@@]1(O[C@H](CO1)CCC)c1c(cc(cc1)Cl)Cl,3.39
4,51,n1c(nc(nc1OC)NC(C)C)NC(C)C,2.57


## RDKit Cleaning

Cleaning rules:

- Parse each raw SMILES with RDKit.
- Drop invalid structures.
- Drop salts or disconnected structures instead of selecting a fragment, preserving a strict one-compound/one-structure modeling set.
- Canonicalize valid single-fragment molecules.
- Drop duplicate canonical SMILES, keeping the first occurrence after sorting by canonical SMILES and compound ID for deterministic output.

In [4]:
def canonicalize_single_fragment(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, "invalid_smiles"
    if len(Chem.GetMolFrags(mol)) != 1:
        return None, "disconnected_or_salt"
    return Chem.MolToSmiles(mol, canonical=True), None


cleaning_records = []
for row in raw_df.itertuples(index=False):
    canonical_smiles, drop_reason = canonicalize_single_fragment(row.SMILES_raw)
    cleaning_records.append({
        "compound_id": row.compound_id,
        "SMILES_raw": row.SMILES_raw,
        "SMILES": canonical_smiles,
        "logKoc": row.logKoc,
        "drop_reason": drop_reason,
    })

cleaning_df = pd.DataFrame(cleaning_records)
structure_drop_count = int(cleaning_df["drop_reason"].notna().sum())
valid_df = cleaning_df[cleaning_df["drop_reason"].isna()].copy()
valid_df = valid_df.sort_values(["SMILES", "compound_id"], kind="mergesort").reset_index(drop=True)
duplicate_mask = valid_df.duplicated(subset="SMILES", keep="first")
duplicate_drop_count = int(duplicate_mask.sum())
clean_df = valid_df.loc[~duplicate_mask, ["SMILES", "logKoc"]].copy()
clean_df = clean_df.sort_values(["logKoc", "SMILES"], kind="mergesort").reset_index(drop=True)

print(f"Total raw parsed compounds: {len(raw_df)}")
print(f"Dropped during RDKit structure cleaning: {structure_drop_count}")
print(f"Dropped as duplicate canonical SMILES: {duplicate_drop_count}")
print(f"Total dropped during cleaning: {len(raw_df) - len(clean_df)}")
print(f"Final cleaned compounds: {len(clean_df)}")
print("Drop reasons:")
print(cleaning_df["drop_reason"].fillna("kept_before_dedup").value_counts().to_string())

assert clean_df["SMILES"].is_unique, "Canonical SMILES are not unique after deduplication"
assert clean_df["logKoc"].notna().all(), "Missing logKoc values remain after cleaning"
display(clean_df.head())


Total raw parsed compounds: 643
Dropped during RDKit structure cleaning: 0
Dropped as duplicate canonical SMILES: 1
Total dropped during cleaning: 1
Final cleaned compounds: 642
Drop reasons:
kept_before_dedup    643


,SMILES,logKoc
0,C=CC=O,-0.31
1,CC(=O)O,0.00
2,CNC(=O)/C=C(\C)OP(=O)(OC)OC,0.00
3,CCO,0.20
4,C1CO1,0.34


## Y-Ranking Train/Test Split

Y-ranking split implementation: sort compounds by `logKoc`, then assign every fifth compound to the external test set. This gives an approximately 80/20 split while spreading test compounds across the response domain.

In [5]:
def y_ranking_split(df: pd.DataFrame, test_stride: int = 5, test_offset: int = 4):
    if test_stride <= 1:
        raise ValueError("test_stride must be greater than 1")
    ranked = df.sort_values(["logKoc", "SMILES"], kind="mergesort").reset_index(drop=True).copy()
    test_mask = ranked.index % test_stride == test_offset
    train = ranked.loc[~test_mask, ["SMILES", "logKoc"]].reset_index(drop=True)
    test = ranked.loc[test_mask, ["SMILES", "logKoc"]].reset_index(drop=True)
    return train, test


train_df, test_df = y_ranking_split(clean_df)

train_smiles = set(train_df["SMILES"])
test_smiles = set(test_df["SMILES"])
overlap = train_smiles.intersection(test_smiles)

print(f"Train rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Train fraction: {len(train_df) / len(clean_df):.3f}")
print(f"Test fraction: {len(test_df) / len(clean_df):.3f}")
print(f"SMILES overlap between Train and Test: {len(overlap)}")
print("\nTrain logKoc range:", (train_df["logKoc"].min(), train_df["logKoc"].max()))
print("Test logKoc range:", (test_df["logKoc"].min(), test_df["logKoc"].max()))

assert len(train_df) + len(test_df) == len(clean_df), "Split does not preserve cleaned row count"
assert len(overlap) == 0, "Data leakage detected: train/test SMILES overlap"
assert abs(len(test_df) / len(clean_df) - 0.20) < 0.01, "Test split is not approximately 20%"

display(train_df.head())
display(test_df.head())


Train rows: 514
Test rows: 128
Train fraction: 0.801
Test fraction: 0.199
SMILES overlap between Train and Test: 0

Train logKoc range: (-0.31, 6.33)
Test logKoc range: (0.34, 6.1)


,SMILES,logKoc
0,C=CC=O,-0.31
1,CC(=O)O,0.00
2,CNC(=O)/C=C(\C)OP(=O)(OC)OC,0.00
3,CCO,0.20
4,C[C@H](O)CO,0.36


,SMILES,logKoc
0,C1CO1,0.34
1,O=c1ccc(=O)[nH][nH]1,0.45
2,C=O,0.56
3,CNC(=O)O/N=C/C(C)(C)S(C)(=O)=O,0.71
4,CN=C=S,0.97


## Save Processed Outputs

Only the generated modeling splits are written. Raw QDB files remain untouched.

In [6]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train_path = PROCESSED_DIR / "train.csv"
test_path = PROCESSED_DIR / "test.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Saved Train CSV: {train_path}")
print(f"Saved Test CSV: {test_path}")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")


Saved Train CSV: /home/jun/Documents/qsar_modeling/data/processed/train.csv
Saved Test CSV: /home/jun/Documents/qsar_modeling/data/processed/test.csv
Train shape: (514, 2)
Test shape: (128, 2)
